In [ ]:
import re
import json
import time
import rasterio
import calendar
import requests
import rasterio
import numpy as np
from pathlib import Path
from datetime import date
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, as_completed

In [ ]:
BASE_URL = "https://download.geoservice.dlr.de/GSP/files/daily/SCE"
OUT_DIR = Path("/mnt/bender/gen_cog/snow_cover_dlr/raw")

N_WORKERS = 4

def make_tasks():
    for year in range(2023, 2024):
        for month in range(5, 6):
            n_days = calendar.monthrange(year, month)[1]

            for day in range(1, n_days + 1):
                date_str = f"{year}{month:02d}{day:02d}"
                filename = f"LIN10A1.{date_str}_wgs84.tif"

                url = f"{BASE_URL}/{year}/{month:02d}/{day:02d}/{filename}"

                out_path = (
                    OUT_DIR
                    / str(year)
                    / f"{month:02d}"
                    / f"{day:02d}"
                    / filename
                )

                yield url, out_path


def download_one(task):
    url, out_path = task

    # Skip file if already downloaded
    if out_path.exists() and out_path.stat().st_size > 0:
        return "Already exists", out_path

    out_path.parent.mkdir(parents=True, exist_ok=True)

    try:
        response = requests.get(url, stream=True, timeout=60)

        if response.status_code == 200:
            temp_path = out_path.with_suffix(out_path.suffix + ".part")

            with open(temp_path, "wb") as f:
                for chunk in response.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        f.write(chunk)

            temp_path.replace(out_path)

            return "Downloaded", out_path

        else:
            return "Missing", url

    except Exception as e:
        return "Failed", f"{url} | {e}"


tasks = list(make_tasks())

with ThreadPoolExecutor(max_workers=N_WORKERS) as executor:
    futures = [executor.submit(download_one, task) for task in tasks]

    for future in as_completed(futures):
        status, info = future.result()
        print(f"{status}: {info}")

In [ ]:
YEAR_START = 2001
YEAR_END = 2002
N_WORKERS = 2

BASE_DIR = Path("/mnt/bender/gen_cog/snow_cover_dlr")
RAW_BASE = BASE_DIR / "raw"
MONTHLY_BASE = BASE_DIR / "monthly"
LOG_DIR = BASE_DIR / "logs"

LOG_DIR.mkdir(parents=True, exist_ok=True)

VERSION = "v20260605"
METHOD = "dlr.gsp.sce"
SUPPORT = "500m"
DEPTH = "s"
BBOX = "go"
EPSG = "epsg.4326"

date_re = re.compile(r"LIN10A1\.(\d{4})(\d{2})(\d{2})_wgs84\.tif$")


def format_runtime(seconds):
    return {
        "runtime_seconds": round(seconds, 2),
        "runtime_minutes": round(seconds / 60, 2),
    }


def monthly_product_name(variable, var_type, year, month):
    start = date(year, month, 1)
    end = date(year, month, calendar.monthrange(year, month)[1])

    return (
        f"{variable}_{METHOD}_{var_type}_{SUPPORT}_{DEPTH}_"
        f"{start:%Y%m%d}_{end:%Y%m%d}_"
        f"{BBOX}_{EPSG}_{VERSION}.tif"
    )


def date_from_name(path):
    m = date_re.search(path.name)
    if not m:
        return None

    yy, mm, dd = m.groups()
    return date(int(yy), int(mm), int(dd))


def dates_in_month(year, month):
    n_days = calendar.monthrange(year, month)[1]
    return [date(year, month, day) for day in range(1, n_days + 1)]


def collect_files_for_year(year):
    RAW_ROOT = RAW_BASE / str(year)
    files_by_month = defaultdict(dict)

    for tif in RAW_ROOT.glob("**/LIN10A1.*_wgs84.tif"):
        dt = date_from_name(tif)

        if dt is None:
            continue

        files_by_month[(dt.year, dt.month)][dt] = tif

    return files_by_month


def process_one_month(task):
    t0 = time.time()

    year = task["year"]
    month = task["month"]
    files = [Path(f) for f in task["files"]]
    missing_dates = task["missing_dates"]

    OUT_DIR = MONTHLY_BASE / str(year)
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    if len(files) == 0:
        elapsed = time.time() - t0

        return {
            "year_month": f"{year}-{month:02d}",
            "status": "skipped_no_files",
            "n_daily_files_used": 0,
            "missing_dates": missing_dates,
            "snow_probability_file": None,
            **format_runtime(elapsed),
        }

    snow_probability_file = OUT_DIR / monthly_product_name(
        "snowcover", "p", year, month
    )

    # Optional count outputs, currently not saved.
    # snow_days_file = OUT_DIR / monthly_product_name(
    #     "snowdays", "count", year, month
    # )
    # valid_days_file = OUT_DIR / monthly_product_name(
    #     "validdays", "count", year, month
    # )

    with rasterio.open(files[0]) as ref:
        profile_probability = ref.profile.copy()
        profile_probability.update(
            dtype="uint8",
            count=1,
            compress="lzw",
            BIGTIFF="YES",
            nodata=255,
        )

        # Count profile kept only if count outputs are needed later.
        # profile_count = ref.profile.copy()
        # profile_count.update(
        #     dtype="uint8",
        #     count=1,
        #     compress="lzw",
        #     BIGTIFF="YES",
        #     nodata=255,
        # )

        with rasterio.open(snow_probability_file, "w", **profile_probability) as prob_out:

            sources = [rasterio.open(f) for f in files]

            try:
                for _, window in ref.block_windows(1):
                    height = int(window.height)
                    width = int(window.width)

                    snow_days = np.zeros((height, width), dtype=np.uint8)
                    valid_land_days = np.zeros((height, width), dtype=np.uint8)

                    for src in sources:
                        arr = src.read(1, window=window)

                        if src.nodata is not None:
                            nodata = arr == src.nodata
                        else:
                            nodata = np.zeros(arr.shape, dtype=bool)

                        # Required DLR Global SnowPack rule:
                        # snow-covered = values >= 64
                        snow = (arr >= 64) & (~nodata)

                        # Valid snow-free land:
                        # bit 32 represents snow-free land.
                        snow_free_land = ((arr & 32) != 0) & (~nodata)

                        valid_land = snow | snow_free_land

                        snow_days += snow.astype(np.uint8)
                        valid_land_days += valid_land.astype(np.uint8)

                    snow_probability = np.full(
                        (height, width),
                        255,
                        dtype=np.uint8,
                    )

                    valid = valid_land_days > 0

                    snow_probability[valid] = np.rint(
                        100.0
                        * snow_days[valid].astype(np.float32)
                        / valid_land_days[valid].astype(np.float32)
                    ).astype(np.uint8)

                    # Count outputs are not saved.
                    # snow_out.write(snow_days, 1, window=window)
                    # valid_out.write(valid_land_days, 1, window=window)

                    prob_out.write(snow_probability, 1, window=window)

            finally:
                for src in sources:
                    src.close()

    elapsed = time.time() - t0

    return {
        "year_month": f"{year}-{month:02d}",
        "status": "processed",
        "n_daily_files_used": len(files),
        "missing_dates": missing_dates,
        "snow_probability_file": str(snow_probability_file),
        # "snow_days_file": str(snow_days_file),
        # "valid_land_days_file": str(valid_days_file),
        **format_runtime(elapsed),
    }


def main():
    total_t0 = time.time()

    tasks = []
    missing_manifest = []

    for year in range(YEAR_START, YEAR_END + 1):
        print(f"Collecting files for {year}")

        files_by_month = collect_files_for_year(year)

        for month in range(1, 13):
            expected_dates = dates_in_month(year, month)
            available = files_by_month.get((year, month), {})

            missing_dates = [
                dt.isoformat()
                for dt in expected_dates
                if dt not in available
            ]

            files = [
                str(available[dt])
                for dt in expected_dates
                if dt in available
            ]

            missing_manifest.append({
                "year_month": f"{year}-{month:02d}",
                "missing_dates": missing_dates,
                "n_missing": len(missing_dates),
            })

            tasks.append({
                "year": year,
                "month": month,
                "files": files,
                "missing_dates": missing_dates,
            })

    monthly_manifest = []

    with ProcessPoolExecutor(max_workers=N_WORKERS) as executor:
        futures = [executor.submit(process_one_month, task) for task in tasks]

        for future in as_completed(futures):
            result = future.result()
            monthly_manifest.append(result)

            print(
                f"{result['status']}: "
                f"{result['year_month']} | "
                f"{result['n_daily_files_used']} daily files | "
                f"{result['runtime_minutes']} min"
            )

    monthly_manifest = sorted(
        monthly_manifest,
        key=lambda x: x["year_month"]
    )

    total_elapsed = time.time() - total_t0

    combined_manifest = {
        "year_start": YEAR_START,
        "year_end": YEAR_END,
        "raw_base_dir": str(RAW_BASE),
        "monthly_base_dir": str(MONTHLY_BASE),
        "software": {
            "script": "process_monthly_gsp_sce_parallel.py",
            "libraries": ["rasterio", "numpy", "concurrent.futures"],
            "parallelization": f"month-level parallel processing with {N_WORKERS} workers",
        },
        "snow_rule": "snow-covered = values >= 64",
        "valid_land_rule": "valid land = snow-covered pixels OR snow-free land pixels with bit 32",
        "output_encoding": "uint8 percentage, 0-100 = snow probability percent, 255 = NoData",
        "missing_dates_by_month": missing_manifest,
        "monthly_composites": monthly_manifest,
        **format_runtime(total_elapsed),
    }

    combined_manifest_file = (
        LOG_DIR
        / f"monthly_processing_manifest_GSP_SCE_{YEAR_START}_{YEAR_END}.json"
    )

    with open(combined_manifest_file, "w") as f:
        json.dump(combined_manifest, f, indent=2)

    print(f"\nCombined manifest written to: {combined_manifest_file}")
    print(f"Total runtime: {total_elapsed / 60:.2f} minutes")


if __name__ == "__main__":
    main()

In [ ]:
# Long-term monthly quantiles

YEAR_START = 2000
YEAR_END = 2025
N_WORKERS = 2

BASE_DIR = Path("/mnt/bender/gen_cog/snow_cover_dlr")
MONTHLY_BASE = BASE_DIR / "monthly"
CLIM_DIR = BASE_DIR / "longterm_monthly"
LOG_DIR = BASE_DIR / "logs"

CLIM_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

VERSION = "v20260605"
METHOD = "dlr.gsp.sce"
SUPPORT = "500m"
DEPTH = "s"
BBOX = "go"
EPSG = "epsg.4326"

SKIP_EXISTING = True

QUANTILES = {
    "p05": 0.05,
    "p50": 0.50,
    "p95": 0.95,
}

monthly_product_re = re.compile(
    r"snowcover_dlr\.gsp\.sce_p_500m_s_(\d{4})(\d{2})\d{2}_\d{8}_go_epsg\.4326_.*\.tif$"
)


def format_runtime(seconds):
    return {
        "runtime_seconds": round(seconds, 2),
        "runtime_minutes": round(seconds / 60, 2),
    }


def longterm_product_name(var_type, month):
    start = date(YEAR_START, month, 1)
    end = date(YEAR_END, month, calendar.monthrange(YEAR_END, month)[1])

    return (
        f"snowcover_{METHOD}_{var_type}_{SUPPORT}_{DEPTH}_"
        f"{start:%Y%m%d}_{end:%Y%m%d}_"
        f"{BBOX}_{EPSG}_{VERSION}.tif"
    )


def collect_monthly_files():
    files_by_calendar_month = defaultdict(list)
    available_years_by_month = defaultdict(list)

    for tif in MONTHLY_BASE.glob("*/snowcover_dlr.gsp.sce_p_500m_s_*.tif"):
        m = monthly_product_re.search(tif.name)

        if not m:
            continue

        yy = int(m.group(1))
        mm = int(m.group(2))

        if YEAR_START <= yy <= YEAR_END:
            files_by_calendar_month[mm].append(str(tif))
            available_years_by_month[mm].append(yy)

    tasks = []

    for month in range(1, 13):
        files = sorted(files_by_calendar_month.get(month, []))
        available_years = sorted(set(available_years_by_month.get(month, [])))
        missing_years = [
            yy for yy in range(YEAR_START, YEAR_END + 1)
            if yy not in available_years
        ]

        tasks.append({
            "month": month,
            "files": files,
            "available_years": available_years,
            "missing_years": missing_years,
        })

    return tasks


def process_one_calendar_month(task):
    t0 = time.time()

    month = task["month"]
    files = [Path(f) for f in task["files"]]
    available_years = task["available_years"]
    missing_years = task["missing_years"]

    if len(files) == 0:
        elapsed = time.time() - t0

        return {
            "calendar_month": f"{month:02d}",
            "status": "skipped_no_files",
            "available_years": available_years,
            "missing_years": missing_years,
            "n_files_used": 0,
            "outputs": {},
            **format_runtime(elapsed),
        }

    out_files = {
        q_name: CLIM_DIR / longterm_product_name(q_name, month)
        for q_name in QUANTILES
    }

    if SKIP_EXISTING and all(
        out_file.exists() and out_file.stat().st_size > 0
        for out_file in out_files.values()
    ):
        elapsed = time.time() - t0

        return {
            "calendar_month": f"{month:02d}",
            "status": "already_exists",
            "available_years": available_years,
            "missing_years": missing_years,
            "n_files_used": len(files),
            "outputs": {
                q_name: str(out_file)
                for q_name, out_file in out_files.items()
            },
            **format_runtime(elapsed),
        }

    with rasterio.open(files[0]) as ref:
        profile = ref.profile.copy()
        profile.update(
            dtype="uint8",
            count=1,
            compress="lzw",
            BIGTIFF="YES",
            nodata=255,
        )

        out_datasets = {
            q_name: rasterio.open(out_file, "w", **profile)
            for q_name, out_file in out_files.items()
        }

        sources = [rasterio.open(f) for f in files]

        try:
            for _, window in ref.block_windows(1):
                arrays = []

                for src in sources:
                    arr = src.read(1, window=window)

                    # Input monthly products:
                    # 0-100 = snow probability percent
                    # 255   = NoData
                    arr = arr.astype(np.float32)
                    arr[arr == 255] = np.nan

                    arrays.append(arr)

                stack = np.stack(arrays, axis=0)

                valid_count = np.sum(~np.isnan(stack), axis=0)
                valid_pixels = valid_count > 0

                for q_name, q_value in QUANTILES.items():
                    q_arr = np.full(
                        stack.shape[1:],
                        255,
                        dtype=np.uint8,
                    )

                    q_calc = np.nanquantile(stack, q_value, axis=0)

                    q_arr[valid_pixels] = np.rint(
                        q_calc[valid_pixels]
                    ).astype(np.uint8)

                    out_datasets[q_name].write(q_arr, 1, window=window)

        finally:
            for src in sources:
                src.close()

            for dst in out_datasets.values():
                dst.close()

    elapsed = time.time() - t0

    return {
        "calendar_month": f"{month:02d}",
        "status": "processed",
        "available_years": available_years,
        "missing_years": missing_years,
        "n_files_used": len(files),
        "outputs": {
            q_name: str(out_file)
            for q_name, out_file in out_files.items()
        },
        **format_runtime(elapsed),
    }


def main():
    total_t0 = time.time()

    tasks = collect_monthly_files()
    manifest = []

    with ProcessPoolExecutor(max_workers=N_WORKERS) as executor:
        futures = [
            executor.submit(process_one_calendar_month, task)
            for task in tasks
        ]

        for future in as_completed(futures):
            result = future.result()
            manifest.append(result)

            print(
                f"{result['status']}: "
                f"month {result['calendar_month']} | "
                f"{result['n_files_used']} files | "
                f"{result['runtime_minutes']} min"
            )

    manifest = sorted(
        manifest,
        key=lambda x: x["calendar_month"]
    )

    total_elapsed = time.time() - total_t0

    longterm_manifest = {
        "year_start": YEAR_START,
        "year_end": YEAR_END,
        "input_monthly_dir": str(MONTHLY_BASE),
        "output_dir": str(CLIM_DIR),
        "input_encoding": "uint8 percentage, 0-100 = monthly snow probability percent, 255 = NoData",
        "output_encoding": "uint8 percentage, 0-100 = long-term monthly snow probability percent, 255 = NoData",
        "quantiles": QUANTILES,
        "software": {
            "script": "longterm_monthly_snow_quantiles_parallel.py",
            "libraries": ["rasterio", "numpy", "concurrent.futures"],
            "parallelization": f"calendar-month parallel processing with {N_WORKERS} workers",
        },
        "monthly_quantiles": manifest,
        **format_runtime(total_elapsed),
    }

    manifest_file = (
        LOG_DIR
        / f"longterm_monthly_quantiles_GSP_SCE_{YEAR_START}_{YEAR_END}.json"
    )

    with open(manifest_file, "w") as f:
        json.dump(longterm_manifest, f, indent=2)

    print(f"\nLong-term monthly quantile manifest written to: {manifest_file}")
    print(f"Total runtime: {total_elapsed / 60:.2f} minutes")


if __name__ == "__main__":
    main()

In [ ]:
from pathlib import Path

folder = Path("/mnt/bender/snow_cover_dlr")

total_size = 0
file_count = 0

for path in folder.rglob("*"):
    if path.is_file():
        try:
            total_size += path.stat().st_size
            file_count += 1
        except FileNotFoundError:
            pass

print(f"Folder: {folder}")
print(f"Files: {file_count}")
print(f"Size: {total_size / 1024**2:.2f} MB")
print(f"Size: {total_size / 1024**3:.2f} GB")

In [ ]:
import json
from pathlib import Path

base_dir = Path("/mnt/bender/gen_cog/snow_cover_dlr/logs")

# Input JSON files
files = [
    base_dir / "monthly_processing_manifest_GSP_SCE_2000_2000.json",
    base_dir / "monthly_processing_manifest_GSP_SCE_2001_2020.json",
    base_dir / "monthly_processing_manifest_GSP_SCE_2021_2025.json",
    base_dir / "monthly_processing_manifest_GSP_SCE_2023_2023.json",  # recent 2023 info
]

# Output file
out_file = base_dir / "monthly_processing_manifest_DLR_GSP_SCE_2000_2025.json"


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


manifests = [load_json(path) for path in files]

# Use the first manifest as a metadata template
combined = {
    "year_start": min(m["year_start"] for m in manifests),
    "year_end": max(m["year_end"] for m in manifests),
    "raw_base_dir": manifests[0].get("raw_base_dir"),
    "monthly_base_dir": manifests[0].get("monthly_base_dir"),
    "software": manifests[0].get("software"),
    "snow_rule": manifests[0].get("snow_rule"),
    "valid_land_rule": manifests[0].get("valid_land_rule"),
    "output_encoding": manifests[0].get("output_encoding"),
}

# Store records by year_month so overlaps can be replaced
missing_by_month = {}
monthly_composites = {}

for manifest in manifests:
    for item in manifest.get("missing_dates_by_month", []):
        missing_by_month[item["year_month"]] = item

    for item in manifest.get("monthly_composites", []):
        monthly_composites[item["year_month"]] = item

# Explicitly overwrite with recent 2023 manifest where available
recent_2023 = load_json(base_dir / "monthly_processing_manifest_GSP_SCE_2023_2023.json")

for item in recent_2023.get("missing_dates_by_month", []):
    missing_by_month[item["year_month"]] = item

for item in recent_2023.get("monthly_composites", []):
    monthly_composites[item["year_month"]] = item

# Sort chronologically
combined["missing_dates_by_month"] = [
    missing_by_month[k] for k in sorted(missing_by_month)
]

combined["monthly_composites"] = [
    monthly_composites[k] for k in sorted(monthly_composites)
]

# Optional runtime summary
combined["runtime_seconds"] = round(
    sum(m.get("runtime_seconds", 0) for m in manifests),
    2
)
combined["runtime_minutes"] = round(combined["runtime_seconds"] / 60, 2)

# Write combined JSON
with open(out_file, "w", encoding="utf-8") as f:
    json.dump(combined, f, indent=2)

print(f"Saved combined manifest to: {out_file}")
print(f"Months included: {len(combined['monthly_composites'])}")
print(f"Date range: {combined['year_start']}–{combined['year_end']}")